# CRISPR-UniPredict: Quick Start Tutorial

This notebook demonstrates how to:
1. Verify installation
2. Load the pretrained model
3. Make predictions
4. Visualize results

**Time to complete:** ~5 minutes

## Step 1: Verify Installation

Check that all required packages are installed.

In [ ]:
# Check Python version
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 9), "Python 3.9+ required"
print("✓ Python version OK")

In [ ]:
# Check required packages
import importlib

packages = ['torch', 'numpy', 'pandas', 'matplotlib', 'seaborn']
for package in packages:
    try:
        mod = importlib.import_module(package)
        version = getattr(mod, '__version__', 'unknown')
        print(f"✓ {package}: {version}")
    except ImportError:
        print(f"✗ {package}: NOT INSTALLED")
        print(f"  Install with: pip install {package}")

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ GPU not available, will use CPU (slower)")

## Step 2: Load Pretrained Model

Load the CRISPR-UniPredict model from checkpoint.

In [ ]:
# Add project to path
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
# Import model
from models.crispr_unipredict import CRISPRUniPredict
from models.encoding import SequenceEncoder

print("✓ Imports successful")

In [ ]:
# Initialize model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = CRISPRUniPredict(device=device)
encoder = SequenceEncoder(device=device)

# Try to load checkpoint
checkpoint_path = project_root / 'models' / 'checkpoints' / 'best.pt'
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded checkpoint from {checkpoint_path}")
else:
    print(f"⚠ Checkpoint not found at {checkpoint_path}")
    print("  Using untrained model (predictions will be random)")

model.eval()
print(f"✓ Model loaded with {model.get_total_params():,} parameters")

## Step 3: Make Predictions

Predict on-target efficiency and off-target risk for example sgRNAs.

In [ ]:
# Example sgRNAs
examples = {
    'High Efficiency': {
        'sgrna': 'GCTAGCTAGCTAGCTAGCTAGCT',
        'target': 'ATGCATGCATGCATGCATGCATG'
    },
    'Medium Efficiency': {
        'sgrna': 'ATGCATGCATGCATGCATGCATG',
        'target': 'GCTAGCTAGCTAGCTAGCTAGCT'
    },
    'Low Efficiency': {
        'sgrna': 'CCGGCCGGCCGGCCGGCCGGCCG',
        'target': 'TTAATTAATTAATTAATTAATTAA'
    }
}

print("Example sgRNAs:")
for name, seq in examples.items():
    print(f"  {name}: {seq['sgrna']}")

In [ ]:
# Make predictions
import pandas as pd

results = []

for name, seqs in examples.items():
    sgrna = seqs['sgrna']
    target = seqs['target']
    
    # Encode sequences
    onehot = encoder.one_hot_encode(sgrna)
    label = encoder.label_encode(sgrna, add_start_token=False)
    
    # Add batch dimension
    onehot = onehot.unsqueeze(0).to(device)
    label = label.unsqueeze(0).to(device)
    
    # Make prediction
    with torch.no_grad():
        on_target, off_target = model(onehot, label, task_type='both')
    
    on_target_score = on_target.item()
    off_target_prob = off_target.item()
    off_target_safety = 1.0 - off_target_prob
    comprehensive_score = on_target_score * off_target_safety
    
    results.append({
        'sgRNA': name,n        'Sequence': sgrna,
        'On-Target': f"{on_target_score:.4f}",
        'Off-Target Risk': f"{off_target_prob:.4f}",
        'Off-Target Safety': f"{off_target_safety:.4f}",
        'Comprehensive Score': f"{comprehensive_score:.4f}"
    })

# Display results
results_df = pd.DataFrame(results)
print("\nPrediction Results:")
print(results_df.to_string(index=False))

## Step 4: Visualize Results

Create visualizations of the predictions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract numeric values
on_target_scores = [float(r['On-Target']) for r in results]
off_target_risks = [float(r['Off-Target Risk']) for r in results]
comp_scores = [float(r['Comprehensive Score']) for r in results]
names = [r['sgRNA'] for r in results]

# Create figure
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: On-Target Scores
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0].bar(names, on_target_scores, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Score', fontweight='bold')
axes[0].set_title('On-Target Efficiency', fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Off-Target Risk
axes[1].bar(names, off_target_risks, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Probability', fontweight='bold')
axes[1].set_title('Off-Target Risk', fontweight='bold')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Comprehensive Score
axes[2].bar(names, comp_scores, color=colors, alpha=0.7, edgecolor='black')
axes[2].set_ylabel('Score', fontweight='bold')
axes[2].set_title('Comprehensive Score', fontweight='bold')
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('quick_start_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to quick_start_results.png")

## Summary

You've successfully:
1. ✓ Verified installation
2. ✓ Loaded the pretrained model
3. ✓ Made predictions
4. ✓ Visualized results

### Next Steps

- **Batch Processing**: See `02_batch_processing.ipynb`
- **Model Training**: See `03_model_training.ipynb`
- **Advanced Analysis**: See `04_advanced_analysis.ipynb`
- **Baseline Comparison**: See `05_comparison_with_baselines.ipynb`

### Troubleshooting

**Issue: Module not found**
```python
# Make sure project root is in path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
```

**Issue: Checkpoint not found**
- Train a model first: `python scripts/train.py --config configs/model_config.yaml`
- Or download pretrained weights from GitHub

**Issue: Out of memory**
- Use CPU instead: `device = 'cpu'`
- Reduce batch size